# CSRA pilot — exploratory version 2

**No external registration or receipt upload is required.** This is an existing-100-item exploratory pilot with an internally frozen plan. It must not be presented as preregistered, independently timestamped, or an untouched confirmatory sample.

Use this updated notebook and matching ZIP. Preserve older folders. Select **Runtime → Change runtime type → GPU**. Code runs in fresh subprocesses so dependency cleanup takes effect without repeatedly restarting the session. This notebook automatically tests and runs all three systems; no manual MODEL edits are needed. No new pilot reports or scientific results were generated during package preparation.

## 1. Load the updated source ZIP

Upload CSRA_Study2_Pilot.zip. The new source directory is separate from the original notebook's /content/CSRA_Study2_Pilot. If the new source directory already exists when resuming, validate and reuse it. Never overwrite frozen source files in place.

In [ ]:
from pathlib import Path, PurePosixPath
import zipfile, io, stat, json, hashlib
BASE = Path('/content/csra_exploratory_v2')
ROOT = BASE/'CSRA_Study2_Pilot'
if not ROOT.exists():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Select only the updated CSRA_Study2_Pilot.zip.')
    name, payload = next(iter(uploaded.items()))
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            p = PurePosixPath(member.filename)
            if p.is_absolute() or '..' in p.parts or not p.parts or p.parts[0] != ROOT.name or stat.S_ISLNK(member.external_attr >> 16):
                raise RuntimeError('Unsafe or unexpected ZIP path: ' + member.filename)
        archive.extractall(BASE)
lock = json.loads((ROOT/'protocol/package_lock.json').read_text())
if lock.get('package_revision') != 'EXPLORATORY_V2_20261019':
    raise RuntimeError('Use the matching updated exploratory ZIP.')
for name, expected in lock['files'].items():
    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('Changed source file: ' + name)
print('Updated source verified:', ROOT)


## 2. Install and check the environment

This removes **both torchvision and torchaudio**, unused for this text-only pilot, before and after installation. It retains the pinned PyTorch CUDA 12.8 build. The final command checks exact package versions, CUDA availability and the Qwen class import without loading weights. Errors stream directly into this cell. Do not rerun an older installation cell afterward.

In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/csra_exploratory_v2/CSRA_Study2_Pilot')
def run_visible(command):
    process = subprocess.Popen(list(map(str, command)), cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='', flush=True)
    result = process.wait()
    if result:
        raise RuntimeError(f'Step failed with exit code {result}; see the full output above.')
run_visible([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'])
run_visible([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run_visible([sys.executable, '-m', 'pip', 'install', '-r', ROOT/'requirements-scoring.txt', '-r', ROOT/'requirements-analysis.txt'])
run_visible([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'])
run_visible([sys.executable, '-u', ROOT/'code/environment_check.py'])


## 3. Mount Drive and restore run variables

The new results directory ends in **exploratory_v2_run1**. Keep run1 and its old preflights untouched. This cell can be rerun after reconnecting. Use one active writer per model. Drive requires interactive authorization in your own account; keep independent downloaded backups after sessions.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, hashlib
ROOT = Path('/content/csra_exploratory_v2/CSRA_Study2_Pilot')
if not ROOT.is_dir():
    raise RuntimeError('Load the updated source ZIP in step 1 first.')
drive.mount('/content/drive')
RUN_DIR = Path('/content/drive/MyDrive/CSRA_Study2_Pilot_runs/exploratory_v2_run1')
RUN_DIR.mkdir(parents=True, exist_ok=True)
def run_visible(command):
    process = subprocess.Popen(list(map(str,command)), cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='', flush=True)
    result = process.wait()
    if result:
        raise RuntimeError(f'Step failed with exit code {result}; see the full output above.')
def run_script(script, *args):
    run_visible([sys.executable, '-u', ROOT/'code'/script, *args])
run_script('run_pilot.py', 'validate')
print('Exploratory results:', RUN_DIR)


## 4. CPU synthetic tests

These tests use deliberately synthetic logits. They do not load real models or generate pilot evidence. The suite checks design, freezes, registered/exploratory separation, interrupted-run recovery and analyses. Temporary test checkpoints are removed afterward.

In [ ]:
run_visible([sys.executable, '-u', ROOT/'tests/selftest.py'])


## 5. Technical preflight for all three models

This automatically runs Qwen, Granite and Phi sequentially; you do not need to change MODEL manually. Only passing records for this exact new package and model/tokenizer revision are reused. Old-package preflight files are not copied or relabeled. These are two fictional prompts, not pilot outcomes. Their timing/memory do not reliably forecast full collection. Each later scoring session also repeats the synthetic test.

In [ ]:
sys.path.insert(0, str(ROOT/'code'))
from execution_context import check_preflight
for model in ('qwen','granite','phi'):
    path = RUN_DIR/f'{model}_preflight.json'
    if path.exists():
        check_preflight(path, model, root=ROOT)
        print(f'{model}: existing passing preflight for this package reused.', flush=True)
    else:
        print(f'Running {model} synthetic technical preflight...', flush=True)
        run_script('run_pilot.py', 'preflight', '--model', model, '--output', path)
print('All three technical preflights are present and valid.')


## 6. Save the internal freeze

Review protocol/EXPLORATORY_AMENDMENT.md and PROTOCOL.md before collecting reports. This cell copies the locked code, inputs and plan into your results folder, recording a **local** UTC timestamp and hash. It requires no website, account, registration receipt or upload. It is not independent evidence of timing. The record is created once and checked on resume; it cannot be newly created over existing report checkpoints. Disclose any report outcomes already seen outside this folder.

In [ ]:
for model in ('qwen','granite','phi'):
    check_preflight(RUN_DIR/f'{model}_preflight.json', model, root=ROOT)
run_script('run_pilot.py', 'freeze', '--output', RUN_DIR)
print('Internal exploratory freeze saved. No external registration has been claimed.')


## 7. Collect or resume all three models

**This starts actual pilot report collection.** It runs 9,600 logical jobs per model, sequentially. Completed/terminal rows are never overwritten; interrupted, uncommitted computations may repeat. If Colab disconnects, restore steps 1 and 3 as needed and rerun this cell in the same folder; reinstall/check the environment if it was reset. Do not run simultaneous copies for the same model or inspect interim hypothesis analyses.

In [ ]:
for model in ('qwen','granite','phi'):
    print(f'Collecting/resuming {model} in EXPLORATORY mode...', flush=True)
    run_script('run_pilot.py', 'run', '--mode', 'exploratory', '--model', model,
               '--output', RUN_DIR, '--preflight', RUN_DIR/f'{model}_preflight.json')
print('All three model runs have finished. Review any terminal failures in the logs.')


## 8. Analyze the complete pilot

The script refuses incomplete job sets, wrong freezes, mixed modes or synthetic checkpoints. It computes the same four planned contrasts using 10,000 paired relation-stratified whole-triple bootstrap draws and 98.75% intervals. The output explicitly identifies an internally frozen **exploratory existing-item pilot**. Failures cause joint item exclusions and must be reported. This step does not run the costly secondary predictive extensions.

In [ ]:
run_script('analyze_pilot.py', '--run-dir', RUN_DIR, '--out', RUN_DIR/'primary_analysis')
import pandas as pd
print((RUN_DIR/'primary_analysis/analysis_status.json').read_text())
display(pd.read_csv(RUN_DIR/'primary_analysis/primary.csv'))


## 9. Planned secondary predictive probes

Optional at this pilot-feasibility stage; complete them before saying the planned secondary predictive analysis is finished. Set RUN_SECONDARY=True to run answer-informed/VSA Ridge at 10,000 draws and GradientBoosting at 2,000 draws. This may take substantial CPU time. No tuning or reduced real-data bootstrap counts are introduced. Leaving this false preserves primary-only status in the analysis records.

In [ ]:
RUN_SECONDARY = False
if RUN_SECONDARY:
    run_script('analyze_pilot.py', '--run-dir', RUN_DIR, '--out', RUN_DIR/'full_analysis', '--secondary')
    print((RUN_DIR/'full_analysis/analysis_status.json').read_text())
else:
    print('Secondary predictive extension not run; primary-only status is retained.')


## 10. Download a backup and return the results

Stop scoring before archiving. Download this ZIP and retain it with the updated source ZIP. It includes checkpoints, technical preflights, the internal freeze and source snapshot, and any completed analyses. Return the result ZIP for review of feasibility, exclusions and precision before deciding on fresh items or manuscript claims. No scientific findings were supplied with the source package.

In [ ]:
from google.colab import files
import shutil
backup = Path('/content/CSRA_Study2_Exploratory_results.zip')
if backup.exists():
    backup.unlink()
shutil.make_archive(str(backup.with_suffix('')), 'zip', root_dir=RUN_DIR)
files.download(str(backup))
